## Feature engineering F1 model Code breakdown



In [ ]:
import pandas as pd
import numpy as np

# Load raw data
results = pd.read_parquet("data/f1_all_results.parquet")
laps = pd.read_parquet("data/f1_all_laps.parquet")
weather = pd.read_parquet("data/f1_all_weather.parquet")


#### **0. Base Race results only**

In [ ]:
races = results[results.SessionType == 'R'].copy()
races = races[['Driver', 'TeamName', 'GridPosition', 'Position', 'Status',
               'Season', 'Round', 'Location', 'Circuit']].copy()

races['GridPosition'] = pd.to_numeric(races['GridPosition'], errors='coerce')
races['Position'] = pd.to_numeric(races['Position'], errors='coerce')

#### 0.a. DNFs get position 21 


In [ ]:
races['Position'] = races['Position'].fillna(21).astype(int)
races['GridPosition'] = races['GridPosition'].fillna(20).astype(int)

#### **Feature 1: Qualifying gap to pole**

Grid position alone only tells you the order, not the spread. Gap-to-pole in seconds captures how much faster the car is. Since qualifying pace is the single strongest predictor of race pace, this is the model's most direct read on raw drivers' speed for the weekend.

In [ ]:
quali = results[results.SessionType == 'Q'][['DriverId', 'Season', 'Round', 'Q1', 'Q2', 'Q3']].copy()

#### Feature 1.a : Best Quali time by each driver (across the three sessions)

In [ ]:
for col in ['Q1', 'Q2', 'Q3']:
    quali[col] = pd.to_timedelta(quali[col]).dt.total_seconds()

quali['QualiBestTime'] = quali[['Q3', 'Q2', 'Q1']].bfill(axis=1).iloc[:, 0]

> Breakdown: We take all three qualifying times, and first convert them to seconds.
Then, we make sure that ALL columns are filled for all drivers with their best time - so even if a driver did not make it Q3, their Q2 time fills the Q3 cell. This is done by using `bfill(axis=1)`, so ampty rows on the left most column is filled with the values on their immediate right. 
The `.iloc[:, 0]` brings the values from the first column, here Q3, which now has the best times clocked by each driver. 

#### Feature 1.b : Gap to Pole

In [ ]:
pole_time = quali.groupby(['Season', 'Round'])['QualiBestTime'].min().reset_index(name='PoleTime')
quali = quali.merge(pole_time, on=['Season', 'Round'], how='left')
quali['QualiGapToPole'] = quali['QualiBestTime'] - quali['PoleTime']

races = races.merge(quali[['DriverId', 'Season', 'Round', 'QualiGapToPole']],
                    on=['DriverId', 'Season', 'Round'], how='left')

#### **Feature 2: Driver Form (avg Finish Last 5 races)**

Recent results carry information that season-long averages wash out: a driver on a hot streak or in a mid-season car upgrade. Rolling over the last 5 races captures current momentum. 

In [ ]:
races = races.sort_values(['DriverId', 'Season', 'Round'])
races['DriverForm'] = (races.groupby('DriverId')['Position']
                       .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

> Breakdown: Here we first sort the values, so that when grouped, we first get all races for a driver together, then, we get a season's race together (all 2023s, all 2024s, etc), and then we get the rounds sorted (round 1,2,3,...) - note that hte rounds would be incremental and not bunched because we first sort byb season and within each season we only have unique rounds. 
Then we group them by Driver, here we will get the neatly organized values for each season and rounds. We `shift(1)` because we do not want to  include the current Races finish into the avg we have available (avoid leaking).


#### **Feature 3: Constructor Form (avg Finish last 5 races)**

In F1 the car matters as much as the driver. Averaging both cars' recent finishes measures the team's current level independent of which driver is in the seat

(Maybe also add **'TeamRaceOvertake'** to identify team form, relative to overtakes typically made at the tracks.)

In [ ]:
team_avg = (races.sort_values(['TeamName', 'Season', 'Round'])
            .groupby(['TeamName', 'Season', 'Round'])['Position']
            .mean()
            .reset_index(name='TeamRaceAvg'))

team_avg['ConstructorForm'] = (team_avg.groupby('TeamName')['TeamRaceAvg']
                               .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

races = races.merge(team_avg[['TeamName', 'Season', 'Round', 'ConstructorForm']],
                    on=['TeamName', 'Season', 'Round'], how='left')

> Breakdown: Here we groupby Team, season AND round because eachteam has two drivers, and we want to take the avg of the two. So we  get how well each team performed at a circuit in a specific race.  
We then determine constructor form similar to driver form, by taking the mean of the last 5 races. 

#### **Feature 4: Circuit Overtaking Difficulty**

Some tracks let cars recover from a bad grid slot (Monza, Bahrain); others lock the order in from lights out (Monaco, Hungary). `Median |finish − grid|` per circuit quantifies that: a high value means positions shuffle a lot, a low value means grid ≈ finish. This tells the model how much to trust *grid position* on each track.

In [ ]:
races['PosChange'] = abs(races['GridPosition'] - races['Position'])

circuit_overtaking = (races.groupby('Circuit')['PosChange']
                      .median()
                      .reset_index(name='CircuitOvertaking'))

races = races.merge(circuit_overtaking, on='Circuit', how='left')

#### **Feature 5: Driver's History at the Circuit**

Some drivers/teams consistently over - under perform at certain tracks. An expanding mean of past finishes at that circuit captures this driver-track affinity that form and quali miss.

In [ ]:
races['DriverCircuitAvg'] = (races.sort_values(['DriverId', 'Circuit', 'Season'])
                             .groupby(['DriverId', 'Circuit'])['Position']
                             .transform(lambda x: x.shift(1).expanding().mean()))

> Breakdown: Here we sort values first to get an easy ascending list of drivers, with circuits and seasons. 
Then we group by driver and circuit, note that the seasons column would be ascending. We calculate the mean, but an expanding mean, meaning that we only see the point in time performance of the driver - excluding the current race. 

#### Feature 5.b: Constructors History at the Circuit

In [ ]:
races['ConstructorCircuitAvg'] = (races.sort_values(['TeamName', 'Circuit', 'Season'])
                             .groupby(['TeamName', 'Circuit'])['Position']
                             .transform(lambda x: x.shift(1).expanding().mean()))

#### **Feature 6: Rain at Race start**

Wet races scramble the expected order: they neutralize car advantages, reward driver skill, and raise incident rates.

In [ ]:
race_weather = (weather[weather.SessionType == 'R']
    .sort_values('Time')
    .groupby(['Season', 'Round'])
    .first()
    .reset_index()
    [['Season', 'Round', 'Rainfall']]
)
race_weather['RainAtStart'] = race_weather['Rainfall'].astype(int)
races = races.merge(race_weather[['Season', 'Round', 'RainAtStart']],
                    on=['Season', 'Round'], how='left')

> Breakdown: WE get the first value, because we want to see the condition at Race start. 

#### **Feature 7: FP2 best lap gap (if available)**

Practice long-run is an early signal of the weekend's pecking order, before qualifying confirms it

In [ ]:
fp2 = laps[laps.SessionType == 'FP2'].copy()
fp2 = fp2.dropna(subset=['LapTime'])
fp2 = fp2[fp2['Deleted'] != True]
fp2['LapSeconds'] = fp2['LapTime'].dt.total_seconds()

#### Feature 7.a: FP2 driver's Best and Overall Fastest

In [ ]:
fp2_best = (fp2.groupby(['Season', 'Round', 'DriverId'])['LapSeconds']
            .min().reset_index(name='FP2BestLap'))

fp2_fastest = (fp2_best.groupby(['Season', 'Round'])['FP2BestLap']
               .min().reset_index(name='FP2SessionBest'))

fp2_best = fp2_best.merge(fp2_fastest, on=['Season', 'Round'], how='left')
fp2_best['FP2GapToFastest'] = fp2_best['FP2BestLap'] - fp2_best['FP2SessionBest']

races = races.merge(fp2_best[['DriverId', 'Season', 'Round', 'FP2GapToFastest']],
                    on=['DriverId', 'Season', 'Round'], how='left')

> Breakdown: Here we first get the best lap clocked by each driver and then we get the fastest overall.

#### **Feature 8: Driver DNF rate**

Reliability and incident-proneness are real, persistent signals — some drivers and cars simply fail to finish more often.

In [ ]:
races['IsDNF'] = races['Status'].apply(lambda x: 0 if x == 'Finished' or str(x).startswith('+') else 1)
races['DNFRate'] = (races.sort_values(['DriverId', 'Season', 'Round'])
                    .groupby('DriverId')['IsDNF']
                    .transform(lambda x: x.shift(1).expanding().mean()))

#### **Cleanup and Save**

In [ ]:
races = races.drop(columns=['Status', 'PosChange', 'IsDNF'])

races.to_parquet("data/f1_model_ready.parquet", index=False)
print(f"Saved {len(races)} rows with columns:")
print(list(races.columns))